In [0]:
import os
import pickle
from pyspark.sql import functions as F
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array

In [0]:
# =========================
# Create DBFS Directory
# =========================
dbfs_path = "workspace.ather"
os.makedirs(dbfs_path, exist_ok=True)

# =========================
# Load Data from Databricks Table (Spark DF)
# =========================
df = spark.table('workspace.ather.ather_table')

In [0]:
# =========================
# Define Target Column (Motor Stress Index proxy)
# =========================
target_column = "Thermal_Load_Index"

# =========================
# Drop non-useful columns
# =========================
drop_cols = ["TIMESTAMP", "Timestamp_1"]
df = df.drop(*drop_cols)

In [0]:
# =========================
# Identify Feature Columns
# =========================
feature_cols = [c for c in df.columns if c != target_column]

In [0]:
# Keep only numeric columns
numeric_cols = [
    c for c, t in df.dtypes
    if c in feature_cols and t in ("int", "bigint", "double", "float", "long", "short", "decimal")
]

# =========================
# Handle Missing Values
# =========================
imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=[c + "_imputed" for c in numeric_cols]
).setStrategy("mean")

In [0]:
# =========================
# Assemble Features
# =========================
assembler = VectorAssembler(
    inputCols=[c + "_imputed" for c in numeric_cols],
    outputCol="features"
)

In [0]:
# =========================
# Scale Features
# =========================
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

In [0]:
# =========================
# Pipeline
# =========================
pipeline = Pipeline(stages=[imputer, assembler, scaler])
pipeline_model = pipeline.fit(df)
processed_df = pipeline_model.transform(df)

In [0]:
# =========================
# Train-Test Split
# =========================
train_df, test_df = processed_df.randomSplit([0.8, 0.2], seed=42)

In [0]:
# =========================
# Convert Vector to Columns
# =========================
feature_size = len(numeric_cols)

train_df = train_df.withColumn("features_array", vector_to_array("scaled_features"))
test_df = test_df.withColumn("features_array", vector_to_array("scaled_features"))

for i in range(feature_size):
    train_df = train_df.withColumn(f"f_{i}", F.col("features_array")[i])
    test_df = test_df.withColumn(f"f_{i}", F.col("features_array")[i])

In [0]:
# =========================
# Final Datasets
# =========================
final_cols = [f"f_{i}" for i in range(feature_size)]

X_train_df = train_df.select(final_cols)
y_train_df = train_df.select(target_column)

X_test_df = test_df.select(final_cols)
y_test_df = test_df.select(target_column)

In [0]:
# =========================
# Save to DBFS (Parquet)
# =========================
X_train_df.write.mode("overwrite").parquet(f"/Volumes/workspace/default/ather/data/X_train.parquet")
X_test_df.write.mode("overwrite").parquet("/Volumes/workspace/default/ather/data/X_test.parquet")
y_train_df.write.mode("overwrite").parquet("/Volumes/workspace/default/ather/data/y_train.parquet")
y_test_df.write.mode("overwrite").parquet("/Volumes/workspace/default/ather/data/y_test.parquet")

In [0]:
# =========================
# Save Pipeline
# =========================
pipeline_model.write().overwrite().save("/Volumes/workspace/default/ather/data/preprocessing_pipeline")